# Install

In [ ]:
!pip3 install pshmodule

In [ ]:
!pip3 install pickle5

In [ ]:
!pip3 install pandas==1.5.0

In [ ]:
!pip3 install swifter

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Load

In [6]:
import sys
sys.path.append('/content/drive/MyDrive/Advertising_By_Personality/src/preprocessing')
print(sys.path)

['/content', '/env/python', '/usr/lib/python39.zip', '/usr/lib/python3.9', '/usr/lib/python3.9/lib-dynload', '', '/usr/local/lib/python3.9/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.9/dist-packages/IPython/extensions', '/root/.ipython', '/content/drive/MyDrive/Advertising_By_Personality/src/preprocessing']


In [7]:
import itertools
import swifter
import config as cfg
import pandas as pd
from tqdm import tqdm
from pshmodule.utils import filemanager as fm

In [8]:
df = fm.load(cfg.origin_data)

extension : .xlsx
Loaded 25000 records from /content/drive/MyDrive/Advertising_By_Personality/data/origin/origin.xlsx


In [9]:
df.rename(columns={'관리번호':'no', '광고 구분':'advertisement_type', '마케팅전략':'marketing_strategy', '일상정보':'daily_information', '시즌정보':'season_information', '분류':'type', '제목':'title', '본문':'content', '납품차수':'degree'}, inplace=True)

In [10]:
df.head()

,no,advertisement_type,marketing_strategy,daily_information,season_information,type,title,content,degree
0,1,NaN,NaN,NaN,NaN,원문,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰,"마지막 2일! [계절밥상 디너/주말 1만원, 런치 5천원 할인 쿠폰] 9월30일까지...",NaN
1,1,할인/쿠폰/혜택,"매력적인숫자(시간,돈)",NaN,NaN,ST,★최대 1만원★ 계절밥상 할인쿠폰,"이.틀.만! VIP 고객 디너&주말 1만원, 런치 5천원 할인 쿠폰 놓치지 마세요!",1차
2,1,할인/쿠폰/혜택,핵심정보만 전달,NaN,NaN,SF,ONLY VIP! 할인쿠폰♬,소중한 이들과 더 든든하게 먹고 싶다면?\\[계절밥상] 디너/주말 1만원+런치 5천...,1차
3,1,할인/쿠폰/혜택,호기심유발(질문),NaN,NaN,NT,이 혜택💰 놓치지 않을 거예요,"[계절밥상] 디너/주말 1O,OOO원, 런치 5,OOO원 할인\\9월 30일까지🚨",1차
4,1,할인/쿠폰/혜택,호기심유발(이모지),NaN,NaN,NF,VIP님들만을 위한 쿠폰 도착💌,계절밥상에서 든든한 한끼🍱 어때요?\\런치/디너/주말 할인 쿠폰(~9/30) 사용하...,1차


# Processing

In [ ]:
df_temp = df.copy()

Convert Emoji to Unicode

In [ ]:
import re

In [ ]:
# 이모지를 Java, JavaScript, JSON 유니코드 코드로 변경
def convert_to_other_unicode(text):
    def unicode_escape(match):
        code_point = ord(match.group(0))
        if code_point <= 0xFFFF:
            return f"\\u{code_point:04x}"
        else:
            high = (code_point - 0x10000) // 0x400 + 0xD800
            low = (code_point - 0x10000) % 0x400 + 0xDC00
            return f"\\u{high:04x}\\u{low:04x}"

    pattern = re.compile(r"([\u2700-\u27BF]|[\U0001F900-\U0001F9FF]|[\U0001F600-\U0001F64F]|[\U0001F300-\U0001F5FF]|[\U0001F680-\U0001F6FF]|[\U0001F1E0-\U0001F1FF])")
    result = pattern.sub(unicode_escape, text)
    result2 = result.replace('❓', '\\u2753').replace('❗', '\\u2757').replace('⭕', '\\u2B55').replace('❌', '\\u274c').replace('⏰', '\\u23F0').replace('⭐', '\\u2B50')
    return result2

In [ ]:
# test
print(convert_to_other_unicode('ㄷㄷㄷ ✨'))
print(convert_to_other_unicode('✋'))
print(convert_to_other_unicode('⭕'))
print(convert_to_other_unicode('❌'))
print(convert_to_other_unicode('⏰'))
print(convert_to_other_unicode('⭐'))
print(convert_to_other_unicode('✈️'))
print(convert_to_other_unicode('💰'))

ㄷㄷㄷ \u2728
\u270b
\u2B55
\u274c
\u23F0
\u2B50
\u2708️
\ud83d\udcb0


In [ ]:
# 이모지를 Java, JavaScript, JSON 유니코드 코드로 변경
df_temp['title_processing'] = df_temp.title.swifter.apply(convert_to_other_unicode)
df_temp['content_processing'] = df_temp.content.swifter.apply(convert_to_other_unicode)

Pandas Apply:   0%|          | 0/25000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/25000 [00:00<?, ?it/s]

In [ ]:
# \\\\ → \\
df_temp['title_processing'] = df_temp.title_processing.swifter.apply(lambda x: x.replace('\\\\', '\\'))
df_temp['content_processing'] = df_temp.content_processing.swifter.apply(lambda x: x.replace('\\\\', '\\'))

Pandas Apply:   0%|          | 0/25000 [00:00<?, ?it/s]

Arrangement

In [ ]:
del df_temp['title']
del df_temp['content']

In [ ]:
df_temp.rename(columns={'title_processing':'title', 'content_processing':'content'}, inplace=True)

In [ ]:
df_temp = df_temp[['no', 'advertisement_type', 'marketing_strategy', 'daily_information', 'season_information', 'type', 'title', 'content', 'degree']]

In [ ]:
df_temp.head()

,no,advertisement_type,marketing_strategy,daily_information,season_information,type,title,content,degree
0,1,NaN,NaN,NaN,NaN,원문,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰,"마지막 2일! [계절밥상 디너/주말 1만원, 런치 5천원 할인 쿠폰] 9월30일까지...",NaN
1,1,할인/쿠폰/혜택,"매력적인숫자(시간,돈)",NaN,NaN,ST,★최대 1만원★ 계절밥상 할인쿠폰,"이.틀.만! VIP 고객 디너&주말 1만원, 런치 5천원 할인 쿠폰 놓치지 마세요!",1차
2,1,할인/쿠폰/혜택,핵심정보만 전달,NaN,NaN,SF,ONLY VIP! 할인쿠폰♬,소중한 이들과 더 든든하게 먹고 싶다면?\[계절밥상] 디너/주말 1만원+런치 5천원...,1차
3,1,할인/쿠폰/혜택,호기심유발(질문),NaN,NaN,NT,이 혜택\ud83d\udcb0 놓치지 않을 거예요,"[계절밥상] 디너/주말 1O,OOO원, 런치 5,OOO원 할인\9월 30일까지\ud...",1차
4,1,할인/쿠폰/혜택,호기심유발(이모지),NaN,NaN,NF,VIP님들만을 위한 쿠폰 도착\ud83d\udc8c,계절밥상에서 든든한 한끼\ud83c\udf71 어때요?\런치/디너/주말 할인 쿠폰(...,1차


# Reshape

In [ ]:
result = []
temp = []

for i in df_temp.iterrows():
  temp.append([i[1]['type'], i[1]['title'] + '\\\\'  + i[1]['content'], i[1]['season_information']])
  if len(temp) == 5:
    index_combinations = list(itertools.combinations(temp, 2))
    for c in index_combinations:
      # c[0][0] 바뀌기 전, c[1][0] 바뀐 후
      result.append([c[0][0], c[1][0], c[0][1], c[1][1], c[1][2]])
    for c in index_combinations:
      result.append([c[1][0], c[0][0], c[1][1], c[0][1], c[0][2]])
    temp = []

In [ ]:
len(result)

100000

In [ ]:
result[45000]

['원문',
 'ST',
 '새해부터 깊카가 챙겨주는 페이백과 경품!\\\\기프트카드를 구매하면 선착순 3천원 페이백 깊카와 구찌,프라다,루이비통 카드지갑까지\\uD83C\\uDFB0',
 '받으면서 시작하는 한 해♥\\\\기프트카드 구매만 해도 \\u2714️선착순 3천원 페이백 \\u2714️구찌, 프라다, 루이비틍 명품 카드지갑 증정\\새해된 기념으로 쏜다!',
 '새해']

In [ ]:
df_result = pd.DataFrame(result, columns=['ctrl1', 'ctrl2', 'input', 'label', 'season'])

In [ ]:
df_result['season'].fillna('', inplace=True)

In [ ]:
df_result.head()

,ctrl1,ctrl2,input,label,season
0,원문,ST,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,"★최대 1만원★ 계절밥상 할인쿠폰\\이.틀.만! VIP 고객 디너&주말 1만원, 런...",
1,원문,SF,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,ONLY VIP! 할인쿠폰♬\\소중한 이들과 더 든든하게 먹고 싶다면?\[계절밥상]...,
2,원문,NT,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,"이 혜택\ud83d\udcb0 놓치지 않을 거예요\\[계절밥상] 디너/주말 1O,O...",
3,원문,NF,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,VIP님들만을 위한 쿠폰 도착\ud83d\udc8c\\계절밥상에서 든든한 한끼\ud...,
4,ST,SF,"★최대 1만원★ 계절밥상 할인쿠폰\\이.틀.만! VIP 고객 디너&주말 1만원, 런...",ONLY VIP! 할인쿠폰♬\\소중한 이들과 더 든든하게 먹고 싶다면?\[계절밥상]...,


# Save

In [ ]:
import json

In [ ]:
temp_dict = [{"ctrl1":row['ctrl1'].strip(), "ctrl2":row['ctrl2'].strip(), "input": row['input'].strip(), "label": row['label'], "season": row['season']} for _, row in df_result.iterrows()]

with open(cfg.train_data, 'w', encoding='utf-8') as f:
    for line in temp_dict:
        json_record = json.dumps(line, ensure_ascii=False)
        f.write(json_record + '\n')

# 이모지로 역치환 - Predict 후에


In [ ]:
import re
import codecs

In [ ]:
# Java, JavaScript, JSON 유니코드 이모지 코드를 찾는 정규식
def convert_to_python_unicode(emoji_code):    
    pattern = re.compile(r"(?:\\u[0-9a-fA-F]{4}){2}")
    
    def surrogate_pair(match):
        code_points = [int(x, 16) for x in match.group(0).split('\\u')[1:]]
        high, low = code_points
        code_point = 0x10000 + ((high - 0xD800) << 10) + (low - 0xDC00)
        return f"\\U{code_point:08x}"

    return pattern.sub(surrogate_pair, emoji_code)

In [ ]:
# 이모지 출력 함수
error_emoji = ['\\U0042089b', '\\udd1a', '\\udf73', '\\U009880e3', '\\U0013bc3d', '\\U0014b80d']

def convert_emojis_in_text(text):
    # 이모지를 찾는 정규 표현식 패턴
    emoji_pattern = r'(\\U[0-9a-fA-F]{8}|\\u[0-9a-fA-F]{4})'
    
    # 이모지를 변환하는 함수
    def convert_emoji(match):
        emoji = match.group(1)
        if emoji in error_emoji:
          return ''
        else:
          return emoji.encode('raw_unicode_escape').decode('unicode-escape')
    
    # 텍스트에서 이모지를 변환
    converted_text = re.sub(emoji_pattern, convert_emoji, text)

    return converted_text

test

In [ ]:
print(convert_emojis_in_text(convert_to_python_unicode("\\u274c잊지말기\\u274c\\\\포인트")))

❌잊지말기❌\\포인트


In [ ]:
# print(convert_emojis_in_text(convert_to_python_unicode('ㄷㄷㄷ \\u2728')))
# print(convert_emojis_in_text(convert_to_python_unicode('ㄷㄷㄷ \\u270b')))
# print(convert_emojis_in_text(convert_to_python_unicode('ㄷㄷㄷ \\u2B55')))
# print(convert_emojis_in_text(convert_to_python_unicode('ㄷㄷㄷ \\u274c')))
# print(convert_emojis_in_text(convert_to_python_unicode('ㄷㄷㄷ \\u23F0')))
# print(convert_emojis_in_text(convert_to_python_unicode('ㄷㄷㄷ \\u2B50')))
# print(convert_emojis_in_text(convert_to_python_unicode('ㄷㄷㄷ \\u2708️')))
# print(convert_emojis_in_text(convert_to_python_unicode('ㄷㄷㄷ \\ud83d\\udcb0')))

ㄷㄷㄷ ✨
ㄷㄷㄷ ✋
ㄷㄷㄷ ⭕
ㄷㄷㄷ ❌
ㄷㄷㄷ ⏰
ㄷㄷㄷ ⭐
ㄷㄷㄷ ✈️
ㄷㄷㄷ 💰


In [ ]:
# Java, JavaScript, JSON 유니코드 이모지 코드를 python 유니코드로 변경
df_result['input_processing'] = df_result.input.swifter.apply(convert_to_python_unicode)
df_result['label_processing'] = df_result.label.swifter.apply(convert_to_python_unicode)

Pandas Apply:   0%|          | 0/100000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/100000 [00:00<?, ?it/s]

In [ ]:
# 이모지 출력하도록 수정
df_result['input_processing'] = df_result.input_processing.swifter.apply(convert_emojis_in_text)
df_result['label_processing'] = df_result.label_processing.swifter.apply(convert_emojis_in_text)

Pandas Apply:   0%|          | 0/100000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/100000 [00:00<?, ?it/s]

In [ ]:
df_result.head()

,ctrl1,ctrl2,input,label,season,input_processing,label_processing
0,원문,ST,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,"★최대 1만원★ 계절밥상 할인쿠폰\\이.틀.만! VIP 고객 디너&주말 1만원, 런...",NaN,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,"★최대 1만원★ 계절밥상 할인쿠폰\\이.틀.만! VIP 고객 디너&주말 1만원, 런..."
1,원문,SF,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,ONLY VIP! 할인쿠폰♬\\소중한 이들과 더 든든하게 먹고 싶다면?\[계절밥상]...,NaN,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,ONLY VIP! 할인쿠폰♬\\소중한 이들과 더 든든하게 먹고 싶다면?\[계절밥상]...
2,원문,NT,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,"이 혜택\ud83d\udcb0 놓치지 않을 거예요\\[계절밥상] 디너/주말 1O,O...",NaN,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,"이 혜택💰 놓치지 않을 거예요\\[계절밥상] 디너/주말 1O,OOO원, 런치 5,O..."
3,원문,NF,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,VIP님들만을 위한 쿠폰 도착\ud83d\udc8c\\계절밥상에서 든든한 한끼\ud...,NaN,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,VIP님들만을 위한 쿠폰 도착💌\\계절밥상에서 든든한 한끼🍱 어때요?\런치/디너/주...
4,ST,SF,"★최대 1만원★ 계절밥상 할인쿠폰\\이.틀.만! VIP 고객 디너&주말 1만원, 런...",ONLY VIP! 할인쿠폰♬\\소중한 이들과 더 든든하게 먹고 싶다면?\[계절밥상]...,NaN,"★최대 1만원★ 계절밥상 할인쿠폰\\이.틀.만! VIP 고객 디너&주말 1만원, 런...",ONLY VIP! 할인쿠폰♬\\소중한 이들과 더 든든하게 먹고 싶다면?\[계절밥상]...


In [ ]:
# \\\\ → \n\n
df_result['input_processing'] = df_result.input_processing.swifter.apply(lambda x: x.replace('\\\\', '\n\n'))
df_result['label_processing'] = df_result.label_processing.swifter.apply(lambda x: x.replace('\\\\', '\n\n'))

In [ ]:
# \\ → \n
df_result['input_processing'] = df_result.input_processing.swifter.apply(lambda x: x.replace('\\', '\n'))
df_result['label_processing'] = df_result.label_processing.swifter.apply(lambda x: x.replace('\\', '\n'))

In [ ]:
df_result.head()

,ctrl1,ctrl2,input,label,season,input_processing,label_processing
0,원문,ST,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,"★최대 1만원★ 계절밥상 할인쿠폰\\이.틀.만! VIP 고객 디너&주말 1만원, 런...",NaN,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\n\n마지막 2일! [계절밥상...,"★최대 1만원★ 계절밥상 할인쿠폰\n\n이.틀.만! VIP 고객 디너&주말 1만원,..."
1,원문,SF,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,ONLY VIP! 할인쿠폰♬\\소중한 이들과 더 든든하게 먹고 싶다면?\[계절밥상]...,NaN,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\n\n마지막 2일! [계절밥상...,ONLY VIP! 할인쿠폰♬\n\n소중한 이들과 더 든든하게 먹고 싶다면?\n[계절...
2,원문,NT,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,"이 혜택\ud83d\udcb0 놓치지 않을 거예요\\[계절밥상] 디너/주말 1O,O...",NaN,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\n\n마지막 2일! [계절밥상...,"이 혜택💰 놓치지 않을 거예요\n\n[계절밥상] 디너/주말 1O,OOO원, 런치 5..."
3,원문,NF,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\\마지막 2일! [계절밥상 디...,VIP님들만을 위한 쿠폰 도착\ud83d\udc8c\\계절밥상에서 든든한 한끼\ud...,NaN,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\n\n마지막 2일! [계절밥상...,VIP님들만을 위한 쿠폰 도착💌\n\n계절밥상에서 든든한 한끼🍱 어때요?\n런치/디...
4,ST,SF,"★최대 1만원★ 계절밥상 할인쿠폰\\이.틀.만! VIP 고객 디너&주말 1만원, 런...",ONLY VIP! 할인쿠폰♬\\소중한 이들과 더 든든하게 먹고 싶다면?\[계절밥상]...,NaN,"★최대 1만원★ 계절밥상 할인쿠폰\n\n이.틀.만! VIP 고객 디너&주말 1만원,...",ONLY VIP! 할인쿠폰♬\n\n소중한 이들과 더 든든하게 먹고 싶다면?\n[계절...
